# ByteFrost — Demand Forecasting Engine (XGBoost)

**Owner:** Agni Pratap Pramanik (AI/ML)  
**Sprint 1:** Aug 26–31, 2026  
**Status:** Prototype complete

This notebook trains an XGBoost regressor on demand data to forecast demand by crop-region-time bucket, producing 7-day and 30-day predictions with confidence intervals. It uses synthetic data while real order data accumulates on the platform.

In [ ]:
# macOS: XGBoost needs OpenMP runtime
import os
os.environ['DYLD_LIBRARY_PATH'] = '/opt/homebrew/opt/libomp/lib:' + os.environ.get('DYLD_LIBRARY_PATH', '')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

print('Imports OK')

## 1. Load synthetic demand data

In [ ]:
df = pd.read_csv('../data/synthetic_demand.csv')
df['date'] = pd.to_datetime(df['date'])
print(f'Rows: {len(df):,}')
print(f'Crops: {df["crop"].nunique()}, Regions: {df["region"].nunique()}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.head()

## 2. Aggregate to daily crop-region demand

In [ ]:
df = (df.groupby(['date', 'crop', 'region'])['demand_kg'].sum().reset_index())
print(f'Aggregated to {len(df):,} daily crop-region rows')
df.head()

## 3. Feature engineering

Lags (t-1, t-7, t-30), rolling averages/volatility, and calendar features (month, week, day-of-week, weekend, festival).

In [ ]:
FEATURE_COLS = [
    'crop_encoded', 'region_encoded', 'month', 'week_of_year', 'day_of_week',
    'is_weekend', 'is_festival', 'lag_1', 'lag_7', 'lag_30',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_30',
]

FESTIVAL_DATES = {
    '2024-10-31', '2025-03-14', '2025-10-20', '2026-03-03', '2026-11-08',
    '2024-08-15', '2025-08-15', '2026-08-15',
    '2024-12-25', '2025-12-25', '2026-12-25',
}

def engineer_features(df):
    df = df.copy()
    df['month'] = df['date'].dt.month
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_festival'] = df['date'].dt.date.isin(FESTIVAL_DATES).astype(int)
    df = df.sort_values(['crop', 'region', 'date']).reset_index(drop=True)
    g = df.groupby(['crop', 'region'])['demand_kg']
    df['lag_1'] = g.shift(1)
    df['lag_7'] = g.shift(7)
    df['lag_30'] = g.shift(30)
    df['rolling_mean_7'] = g.transform(lambda x: x.rolling(7, min_periods=1).mean())
    df['rolling_std_7'] = g.transform(lambda x: x.rolling(7, min_periods=1).std())
    df['rolling_mean_30'] = g.transform(lambda x: x.rolling(30, min_periods=1).mean())
    df['crop_encoded'] = df['crop'].astype('category').cat.codes
    df['region_encoded'] = df['region'].astype('category').cat.codes
    return df

df = engineer_features(df)
df = df.dropna(subset=['lag_1', 'lag_7', 'lag_30']).reset_index(drop=True)
print(f'Rows after dropping rows without lag features: {len(df):,}')
df[['date', 'crop', 'region', 'demand_kg'] + FEATURE_COLS].head()

## 4. Time-based train/test split

In [ ]:
max_date = df['date'].max()
cutoff = max_date - pd.DateOffset(months=3)
train = df[df['date'] < cutoff]
test = df[df['date'] >= cutoff]
print(f'Train: {len(train):,} rows ({train["date"].min().date()} to {train["date"].max().date()})')
print(f'Test:  {len(test):,} rows ({test["date"].min().date()} to {test["date"].max().date()})')

X_train = train[FEATURE_COLS]
y_train = np.log1p(train['demand_kg'])  # log-transform to stabilize variance
X_test = test[FEATURE_COLS]
y_test = test['demand_kg']

## 5. Train XGBoost

In [ ]:
model = XGBRegressor(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
)
model.fit(X_train, y_train, eval_set=[(X_test, np.log1p(y_test))], verbose=False)
print('Training complete')

## 6. Evaluate (next-day demand)

In [ ]:
preds = np.expm1(model.predict(X_test))
rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
print(f'RMSE: {rmse:.2f} kg')
print(f'MAE:  {mae:.2f} kg')
print(f'MAPE: {mape:.2f}%')
print(f'Mean demand: {y_test.mean():.2f} kg')
print(f'MAE as % of mean: {mae / y_test.mean() * 100:.2f}%')

## 7. Feature importance

In [ ]:
imp = sorted(zip(FEATURE_COLS, model.feature_importances_), key=lambda x: x[1], reverse=True)
plt.figure(figsize=(8, 5))
sns.barplot(x=[i[1] for i in imp[:10]], y=[i[0] for i in imp[:10]])
plt.title('XGBoost Feature Importance (Demand Forecasting)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Save model for serving

In [ ]:
import joblib, json, os
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/demand_forecast_xgb.joblib')
crop_map = dict(enumerate(df['crop'].astype('category').cat.categories))
region_map = dict(enumerate(df['region'].astype('category').cat.categories))
meta = {
    'feature_cols': FEATURE_COLS,
    'crop_map': {str(k): v for k, v in crop_map.items()},
    'region_map': {str(k): v for k, v in region_map.items()},
    'metrics': {'rmse': float(rmse), 'mae': float(mae), 'mape': float(mape)},
    'trained_on': 'synthetic_demand.csv',
    'horizons': [7, 30],
}
with open('../models/demand_forecast_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Model + metadata saved to ../models/')